# Member 1: Data Engineering & Preprocessing
**Dataset:** `weatherAUS.csv` | **Target:** `RainTomorrow`

Put `weatherAUS.csv` next to this notebook and run all cells. Outputs go to `member1_outputs/`.

In [1]:
import json
import os
import platform

import joblib
import numpy as np
import pandas as pd
import sklearn
import imblearn
from imblearn.over_sampling import SMOTENC
from sklearn.compose import ColumnTransformer
from sklearn.impute import MissingIndicator, SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, OrdinalEncoder, StandardScaler

DATA_PATH = 'weatherAUS.csv'
OUT = 'member1_outputs'
SEED = 42
os.makedirs(OUT, exist_ok=True)

## 1. Data validation

In [2]:
raw = pd.read_csv(DATA_PATH, parse_dates=['Date'])
print('Shape:', raw.shape)
print('Date range:', raw.Date.min().date(), 'to', raw.Date.max().date())
print('Locations:', raw.Location.nunique())
print()
print(raw.dtypes.to_string())

Shape: (145460, 23)
Date range: 2007-11-01 to 2017-06-25
Locations: 49

Date             datetime64[us]
Location                    str
MinTemp                 float64
MaxTemp                 float64
Rainfall                float64
Evaporation             float64
Sunshine                float64
WindGustDir                 str
WindGustSpeed           float64
WindDir9am                  str
WindDir3pm                  str
WindSpeed9am            float64
WindSpeed3pm            float64
Humidity9am             float64
Humidity3pm             float64
Pressure9am             float64
Pressure3pm             float64
Cloud9am                float64
Cloud3pm                float64
Temp9am                 float64
Temp3pm                 float64
RainToday                   str
RainTomorrow                str


In [3]:
oktas = ['Cloud9am', 'Cloud3pm']
humidity = ['Humidity9am', 'Humidity3pm']
checks = {
    'duplicate rows': int(raw.duplicated().sum()),
    'duplicate Date + Location': int(raw.duplicated(['Date', 'Location']).sum()),
    'MinTemp above MaxTemp': int((raw.MinTemp > raw.MaxTemp).sum()),
    'negative Rainfall': int((raw.Rainfall < 0).sum()),
    'humidity outside 0-100': int(((raw[humidity] < 0) | (raw[humidity] > 100)).sum().sum()),
    'humidity exactly 0': int((raw[humidity] == 0).sum().sum()),
    'cloud above 8 oktas': int((raw[oktas] > 8).sum().sum()),
    'wind speed above gust speed': int(((raw.WindSpeed9am > raw.WindGustSpeed) | (raw.WindSpeed3pm > raw.WindGustSpeed)).sum()),
    'RainToday disagrees with Rainfall > 1mm': int((((raw.Rainfall > 1) & (raw.RainToday == 'No')) | ((raw.Rainfall <= 1) & (raw.RainToday == 'Yes'))).sum()),
}
pd.Series(checks, name='rows or values').to_frame()

,rows or values
duplicate rows,0
duplicate Date + Location,0
MinTemp above MaxTemp,0
negative Rainfall,0
humidity outside 0-100,0
humidity exactly 0,5
cloud above 8 oktas,3
wind speed above gust speed,5
RainToday disagrees with Rainfall > 1mm,0


## 2. Missing values
Several high-missing columns are absent for entire locations, so these gaps are mostly stations without that instrument rather than random blanks.

In [4]:
missing = pd.DataFrame({
    'missing_pct': (raw.isna().mean() * 100).round(1),
    'locations_with_no_data': raw.groupby('Location').apply(lambda g: g.isna().all(), include_groups=False).sum(),
}).sort_values('missing_pct', ascending=False)
missing[missing.missing_pct > 0]

,missing_pct,locations_with_no_data
Sunshine,48.0,19.0
Evaporation,43.2,16.0
Cloud3pm,40.8,12.0
Cloud9am,38.4,12.0
Pressure9am,10.4,4.0
Pressure3pm,10.3,4.0
WindDir9am,7.3,0.0
WindGustDir,7.1,2.0
WindGustSpeed,7.1,2.0
Humidity3pm,3.1,0.0


## 3. Cleaning
- Rows without `RainTomorrow` are dropped, because the target cannot be filled in without inventing labels.
- Cloud above 8 oktas and humidity of exactly 0 are impossible or sensor errors, so they become missing.
- `Date` becomes `Month`. `Year` is left out because future data will not contain past years.

In [5]:
clean = raw.copy()
clean[oktas] = clean[oktas].where(clean[oktas] <= 8)
clean[humidity] = clean[humidity].where(clean[humidity] > 0)
clean = clean.dropna(subset=['RainTomorrow']).reset_index(drop=True)
clean['Month'] = clean.Date.dt.month
clean['Year'] = clean.Date.dt.year
print('Rows dropped for missing target:', len(raw) - len(clean))
print('Rows remaining:', len(clean))
print(clean.RainTomorrow.value_counts(normalize=True).round(4).to_string())

Rows dropped for missing target: 3267
Rows remaining: 142193
RainTomorrow
No     0.7758
Yes    0.2242


## 4. Train / validation / test split
Stratified 70/15/15. Splitting happens before any fitting, so medians, outlier limits, encoders and the scaler learn from the training set only.

Limitation: a random split places neighbouring days from one station in different sets, so scores may be slightly optimistic compared with forecasting future dates.

In [6]:
LOG_COL = ['Rainfall']
CAP_COLS = ['MinTemp', 'MaxTemp', 'Evaporation', 'Sunshine', 'WindGustSpeed', 'WindSpeed9am', 'WindSpeed3pm',
            'Humidity9am', 'Humidity3pm', 'Pressure9am', 'Pressure3pm', 'Cloud9am', 'Cloud3pm', 'Temp9am', 'Temp3pm']
FLAG_COLS = ['Sunshine', 'Evaporation', 'Cloud9am', 'Cloud3pm']
CAT_COLS = ['Location', 'WindGustDir', 'WindDir9am', 'WindDir3pm']
INPUT_COLS = LOG_COL + CAP_COLS + CAT_COLS + ['RainToday', 'Month']

clean[CAT_COLS + ['RainToday']] = clean[CAT_COLS + ['RainToday']].astype(object).where(clean[CAT_COLS + ['RainToday']].notna(), np.nan)

idx_train, idx_temp = train_test_split(clean.index, test_size=0.30, stratify=clean.RainTomorrow, random_state=SEED)
idx_val, idx_test = train_test_split(idx_temp, test_size=0.50, stratify=clean.loc[idx_temp, 'RainTomorrow'], random_state=SEED)

clean['split'] = 'train'
clean.loc[idx_val, 'split'] = 'val'
clean.loc[idx_test, 'split'] = 'test'

X_train_raw, X_val_raw, X_test_raw = (clean.loc[i, INPUT_COLS] for i in (idx_train, idx_val, idx_test))
y_train, y_val, y_test = ((clean.loc[i, 'RainTomorrow'] == 'Yes').astype(int).rename('RainTomorrow') for i in (idx_train, idx_val, idx_test))

pd.DataFrame({
    'rows': [len(y_train), len(y_val), len(y_test)],
    'share': np.round([len(y_train) / len(clean), len(y_val) / len(clean), len(y_test) / len(clean)], 3),
    'rain_rate': np.round([y_train.mean(), y_val.mean(), y_test.mean()], 4),
}, index=['train', 'val', 'test'])

,rows,share,rain_rate
train,99535,0.70,0.2242
val,21329,0.15,0.2242
test,21329,0.15,0.2242


## 5. Outliers
- **IQR capping** (1.5 x IQR, limits from training data) for the numeric columns below. Extreme values are kept at the limit rather than removed.
- **`Rainfall` uses a log transform instead.** About 63% of days have no rain, so the IQR upper limit is near 2 mm and capping would erase the difference between light rain and a storm.

In [7]:
q1, q3 = X_train_raw[CAP_COLS].quantile(0.25), X_train_raw[CAP_COLS].quantile(0.75)
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr

outliers = pd.DataFrame({
    'lower_cap': lower.round(2),
    'upper_cap': upper.round(2),
    'below': (X_train_raw[CAP_COLS] < lower).sum(),
    'above': (X_train_raw[CAP_COLS] > upper).sum(),
})
outliers['capped_pct'] = ((outliers.below + outliers.above) / len(X_train_raw) * 100).round(2)
outliers

,lower_cap,upper_cap,below,above,capped_pct
MinTemp,-6.35,30.85,32,3,0.04
MaxTemp,2.30,43.90,198,96,0.30
Evaporation,-4.60,14.60,0,1388,1.39
Sunshine,-3.65,19.15,0,0,0.00
WindGustSpeed,5.50,73.50,0,2103,2.11
WindSpeed9am,-11.00,37.00,0,1192,1.20
WindSpeed3pm,-3.50,40.50,0,1740,1.75
Humidity9am,18.00,122.00,1024,0,1.03
Humidity3pm,-6.50,109.50,0,0,0.00
Pressure9am,998.90,1036.50,671,226,0.90


## 6. Preprocessing pipeline
| Columns | Steps |
|---|---|
| `Rainfall` | median fill, log(1 + x), standard scale |
| Other numeric | median fill, IQR cap, standard scale |
| `Sunshine`, `Evaporation`, `Cloud9am`, `Cloud3pm` | also get a was-missing flag |
| `Location`, wind directions | missing becomes `Unknown`, one-hot encode |
| `Month` | one-hot encode |
| `RainToday` | most common value fill, No = 0, Yes = 1 |

In [8]:
preprocessor = ColumnTransformer([
    ('rain', Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('log', FunctionTransformer(np.log1p, feature_names_out='one-to-one')),
        ('scale', StandardScaler()),
    ]), LOG_COL),
    ('num', Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('cap', FunctionTransformer(np.clip, kw_args={'a_min': lower.values, 'a_max': upper.values}, feature_names_out='one-to-one')),
        ('scale', StandardScaler()),
    ]), CAP_COLS),
    ('flag', MissingIndicator(features='all'), FLAG_COLS),
    ('cat', Pipeline([
        ('impute', SimpleImputer(strategy='constant', fill_value='Unknown')),
        ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
    ]), CAT_COLS),
    ('month', OneHotEncoder(categories=[list(range(1, 13))], handle_unknown='ignore', sparse_output=False), ['Month']),
    ('rain_today', Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('encode', OrdinalEncoder(categories=[['No', 'Yes']])),
    ]), ['RainToday']),
], verbose_feature_names_out=False)

def to_frame(array, index):
    return pd.DataFrame(np.asarray(array, dtype=float), columns=feature_names, index=index)

train_array = preprocessor.fit_transform(X_train_raw)
feature_names = list(preprocessor.get_feature_names_out())
X_train = to_frame(train_array, X_train_raw.index)
X_val = to_frame(preprocessor.transform(X_val_raw), X_val_raw.index)
X_test = to_frame(preprocessor.transform(X_test_raw), X_test_raw.index)
print('Features:', len(feature_names))
print('Train / val / test:', X_train.shape, X_val.shape, X_test.shape)
print('Missing values after preprocessing:', int(X_train.isna().sum().sum() + X_val.isna().sum().sum() + X_test.isna().sum().sum()))

Features: 133
Train / val / test: (99535, 133) (21329, 133) (21329, 133)
Missing values after preprocessing: 0


## 7. Class imbalance
Only about 22% of days are followed by rain. SMOTE is applied to the training set only, because validation and test must keep the real class ratio.

`SMOTENC` is used because plain SMOTE would create impossible category values, such as 0.4 of a location. One-hot groups are collapsed to single categories, resampled, then expanded again.

In [9]:
cat_ohe = preprocessor.named_transformers_['cat'].named_steps['ohe']
groups = {col: [f'{col}_{c}' for c in cats] for col, cats in zip(CAT_COLS, cat_ohe.categories_)}
groups['Month'] = [f'Month_{m}' for m in range(1, 13)]
single_flags = [c for c in feature_names if c.startswith('missingindicator_')] + ['RainToday']
numeric_cols = LOG_COL + CAP_COLS

collapsed = X_train[numeric_cols + single_flags].copy()
for name, cols in groups.items():
    collapsed[f'__{name}'] = X_train[cols].values.argmax(axis=1)

categorical = [c for c in collapsed.columns if c not in numeric_cols]
smote = SMOTENC(categorical_features=[collapsed.columns.get_loc(c) for c in categorical], random_state=SEED)
resampled, y_train_smote = smote.fit_resample(collapsed, y_train)

X_train_smote = resampled[numeric_cols + single_flags].copy()
for name, cols in groups.items():
    codes = resampled[f'__{name}'].astype(int).values
    expanded = pd.DataFrame(np.eye(len(cols))[codes], columns=cols, index=resampled.index)
    X_train_smote = pd.concat([X_train_smote, expanded], axis=1)
X_train_smote = X_train_smote[feature_names]
y_train_smote = y_train_smote.rename('RainTomorrow')

print('Before SMOTE:', y_train.value_counts().sort_index().to_dict())
print('After SMOTE: ', y_train_smote.value_counts().sort_index().to_dict())

Before SMOTE: {0: 77221, 1: 22314}
After SMOTE:  {0: 77221, 1: 77221}


### Is SMOTE necessary?
A quick logistic regression check on the validation set. This only informs the imbalance decision; model selection and tuning belong to Member 3.

In [10]:
def score(X, y, **kwargs):
    model = LogisticRegression(max_iter=2000, **kwargs).fit(X, y)
    proba = model.predict_proba(X_val)[:, 1]
    pred = (proba >= 0.5).astype(int)
    return {'ROC-AUC': roc_auc_score(y_val, proba), 'Precision': precision_score(y_val, pred),
            'Recall': recall_score(y_val, pred), 'F1': f1_score(y_val, pred)}

imbalance_check = pd.DataFrame({
    'No resampling': score(X_train, y_train),
    'class_weight=balanced': score(X_train, y_train, class_weight='balanced'),
    'SMOTENC': score(X_train_smote, y_train_smote),
}).T.round(4)
imbalance_check

,ROC-AUC,Precision,Recall,F1
No resampling,0.8693,0.7337,0.5186,0.6077
class_weight=balanced,0.8700,0.5240,0.7754,0.6254
SMOTENC,0.8682,0.5286,0.7687,0.6264


## 8. Save outputs

In [11]:
clean.to_csv(f'{OUT}/weather_clean.csv', index=False)
for name, X, y in [('train', X_train, y_train), ('val', X_val, y_val), ('test', X_test, y_test), ('train_smote', X_train_smote, y_train_smote)]:
    X.to_csv(f'{OUT}/X_{name}.csv.gz', index=False, float_format='%.6f')
    y.to_csv(f'{OUT}/y_{name}.csv', index=False)
joblib.dump(preprocessor, f'{OUT}/preprocessor.joblib')

summary = {
    'target': 'RainTomorrow (No=0, Yes=1)',
    'seed': SEED,
    'raw_rows': int(len(raw)),
    'clean_rows': int(len(clean)),
    'split_rows': {'train': int(len(y_train)), 'val': int(len(y_val)), 'test': int(len(y_test)), 'train_smote': int(len(y_train_smote))},
    'input_columns': INPUT_COLS,
    'output_features': feature_names,
    'validation_checks': checks,
    'iqr_caps': {c: [float(lower[c]), float(upper[c])] for c in CAP_COLS},
    'imbalance_check_val': imbalance_check.to_dict('index'),
    'versions': {'python': platform.python_version(), 'pandas': pd.__version__, 'numpy': np.__version__,
                 'scikit-learn': sklearn.__version__, 'imbalanced-learn': imblearn.__version__, 'joblib': joblib.__version__},
}
with open(f'{OUT}/preprocessing_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2)

for f in sorted(os.listdir(OUT)):
    print(f'{f:32s} {os.path.getsize(os.path.join(OUT, f)) / 1e6:8.2f} MB')

README.md                            0.00 MB
X_test.csv.gz                        1.46 MB
X_train.csv.gz                       6.83 MB
X_train_smote.csv.gz                11.26 MB
X_val.csv.gz                         1.46 MB
preprocessing_summary.json           0.01 MB
preprocessor.joblib                  0.01 MB
weather_clean.csv                   17.27 MB
y_test.csv                           0.06 MB
y_train.csv                          0.30 MB
y_train_smote.csv                    0.46 MB
y_val.csv                            0.06 MB


## 9. Verify the outputs

In [12]:
loaded = joblib.load(f'{OUT}/preprocessor.joblib')
reloaded_val = pd.read_csv(f'{OUT}/X_val.csv.gz')
y_reloaded = pd.read_csv(f'{OUT}/y_val.csv').RainTomorrow

assert set(idx_train).isdisjoint(idx_val) and set(idx_train).isdisjoint(idx_test) and set(idx_val).isdisjoint(idx_test)
assert len(idx_train) + len(idx_val) + len(idx_test) == len(clean)
assert list(reloaded_val.columns) == feature_names and len(reloaded_val) == len(y_reloaded)
assert np.allclose(reloaded_val.values, np.asarray(loaded.transform(X_val_raw), dtype=float), atol=1e-5)
assert not X_train.isna().any().any() and not X_val.isna().any().any() and not X_test.isna().any().any() and not X_train_smote.isna().any().any()
assert abs(y_train.mean() - y_val.mean()) < 0.005 and abs(y_train.mean() - y_test.mean()) < 0.005
assert y_train_smote.mean() == 0.5
assert np.allclose(X_train[numeric_cols].mean(), 0, atol=1e-8) and np.allclose(X_train[numeric_cols].std(ddof=0), 1, atol=1e-8)

one_hot = [c for cols in groups.values() for c in cols]
assert set(np.unique(X_train_smote[one_hot + single_flags].values)) <= {0.0, 1.0}
assert (X_train_smote[[c for c in groups['Location']]].sum(axis=1) == 1).all()
assert (X_train_smote.iloc[:len(X_train)].values == X_train.values).all()
print('All checks passed')

All checks passed


## 10. Using the preprocessor on new input
The saved preprocessor takes these raw columns. `Date` is not an input, so pass `Month` instead. Unmeasured values can be left as `np.nan`.

In [13]:
example = pd.DataFrame([{
    'Rainfall': 0.0, 'MinTemp': 13.4, 'MaxTemp': 22.9, 'Evaporation': np.nan, 'Sunshine': np.nan,
    'WindGustSpeed': 44.0, 'WindSpeed9am': 20.0, 'WindSpeed3pm': 24.0, 'Humidity9am': 71.0, 'Humidity3pm': 22.0,
    'Pressure9am': 1007.7, 'Pressure3pm': 1007.1, 'Cloud9am': 8.0, 'Cloud3pm': np.nan, 'Temp9am': 16.9, 'Temp3pm': 21.8,
    'Location': 'Albury', 'WindGustDir': 'W', 'WindDir9am': 'W', 'WindDir3pm': 'WNW', 'RainToday': 'No', 'Month': 12,
}])[INPUT_COLS]
example[CAT_COLS + ['RainToday']] = example[CAT_COLS + ['RainToday']].astype(object)
print(loaded.transform(example).shape)

(1, 133)
